In [ ]:
python <<'PY'
from pathlib import Path
import duckdb
import pandas as pd

MANIFEST = Path("/share/storage/monade/rota/results/official_labeler_manifest_paths_mar2026.txt")
OFFICIAL_LABELER = "did:plc:ar7c4by46qjdydhdevvrndac"

paths = [
    line.strip()
    for line in MANIFEST.read_text().splitlines()
    if line.strip() and not line.strip().startswith("#")
]

if not paths:
    raise RuntimeError("Manifest vuoto.")

sql_paths = "[" + ", ".join(
    "'" + p.replace("'", "''") + "'" for p in paths
) + "]"

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

con = duckdb.connect()

result = con.execute(f"""
    SELECT
        CASE
            WHEN uri LIKE 'did:%'
                THEN 'account_level'
            WHEN uri LIKE 'at://%/app.bsky.feed.post/%'
                THEN 'post_level'
            WHEN uri LIKE 'at://%'
                THEN 'other_record_level'
            ELSE 'unknown_level'
        END AS label_level,

        val AS label_type,

        COUNT(*) FILTER (
            WHERE lower(neg) = 'false'
        ) AS n_label_added,

        COUNT(*) FILTER (
            WHERE lower(neg) = 'true'
        ) AS n_label_removed,

        COUNT(DISTINCT uri) FILTER (
            WHERE lower(neg) = 'false'
        ) AS n_distinct_labeled_objects

    FROM read_csv(
        {sql_paths},
        header = true,
        delim = ',',
        all_varchar = true
    )

    WHERE ts >= '2026-03-01'
      AND ts <  '2026-04-01'
      AND src = '{OFFICIAL_LABELER}'

    GROUP BY label_level, val
    ORDER BY label_level, n_label_added DESC, label_type;
""").fetchdf()

summary = (
    result.groupby("label_level", as_index=False)
    .agg(
        n_label_types=("label_type", "nunique"),
        n_label_added=("n_label_added", "sum"),
        n_label_removed=("n_label_removed", "sum"),
        n_distinct_labeled_objects=("n_distinct_labeled_objects", "sum")
    )
)

account_labels = result[result["label_level"] == "account_level"].drop(columns="label_level")
post_labels = result[result["label_level"] == "post_level"].drop(columns="label_level")
other_labels = result[result["label_level"] == "other_record_level"].drop(columns="label_level")
unknown_labels = result[result["label_level"] == "unknown_level"].drop(columns="label_level")

print(f"\nFile letti dal manifest: {len(paths):,}")
print(f"Labeler filtrato:         {OFFICIAL_LABELER}")

print("\n RIEPILOGO PER LIVELLO DELLA LABEL ")
print(summary.to_string(index=False))

print("\n ACCOUNT-LEVEL LABELS ")
if account_labels.empty:
    print("Nessuna label direttamente applicata ad account.")
else:
    print(account_labels.to_string(index=False))

print("\n POST-LEVEL LABELS ")
if post_labels.empty:
    print("Nessuna label applicata a post.")
else:
    print(post_labels.to_string(index=False))

print("\n ALTRE RECORD-LEVEL LABELS, NON POST ")
if other_labels.empty:
    print("Nessuna label applicata ad altri tipi di record.")
else:
    print(other_labels.to_string(index=False))

if not unknown_labels.empty:
    print("\n URI NON CLASSIFICATI ")
    print(unknown_labels.to_string(index=False))

con.close()
PY

#ACCOUNT-LEVEL LABELS 
#       label_type  n_label_added  n_label_removed  n_distinct_labeled_objects
#     needs-review         243983            42029                      137411
#        !takedown           159161            3705                     158803
#             spam              1005            214                       942
#         !suspend             821             797                        799
#           sexual               309            35                        309
#            !hide                104           13                        104
#            !warn                54            6                         53
#             rude                30             0                        30
#sexual-figurative              29               3                        27
#    impersonation              24              11                        24
#       intolerant               9               1                        9
#             porn              8                2                        8 
#        self-harm              5                1                        5
#           nudity              2                1                        2
#    graphic-media              1                1                        1
#            rumor               1               0                        1


In [ ]:
python <<'PY'
from pathlib import Path
import duckdb
import pandas as pd


# CONFIGURAZIONE


MANIFEST = Path(
    "/share/storage/monade/rota/results/"
    "official_labeler_manifest_paths_mar2026.txt"
)

OFFICIAL_LABELER = "did:plc:ar7c4by46qjdydhdevvrndac"

POSITIVE_PATH = Path(
    "/share/storage/monade/rota/results/blocks_mar2026/"
    "positive_blocks_analysis_10d_mar2026_v3.parquet"
)

LABEL_TYPES = [
    "porn",
    "sexual",
    "nudity",
    "sexual-figurative",
    "rude",
]


# PATH E IMPOSTAZIONI


if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest non trovato: {MANIFEST}")

if not POSITIVE_PATH.exists():
    raise FileNotFoundError(f"Dataset positivi non trovato: {POSITIVE_PATH}")

paths = [
    line.strip()
    for line in MANIFEST.read_text().splitlines()
    if line.strip() and not line.strip().startswith("#")
]

if not paths:
    raise RuntimeError("Manifest vuoto.")

sql_paths = "[" + ", ".join(
    "'" + p.replace("'", "''") + "'" for p in paths
) + "]"

positive_path_sql = str(POSITIVE_PATH).replace("'", "''")
labels_sql = ", ".join("'" + x.replace("'", "''") + "'" for x in LABEL_TYPES)

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET preserve_insertion_order TO false")


# POSITIVI


con.execute(f"""
CREATE TEMP TABLE positives AS
SELECT DISTINCT
    CAST(did AS VARCHAR) AS did,
    CAST(event_time AS TIMESTAMP) AS event_time
FROM read_parquet('{positive_path_sql}')
WHERE did IS NOT NULL
  AND event_time IS NOT NULL;
""")

positive_check = con.execute("""
SELECT
    COUNT(*) AS n_positive_observations,
    COUNT(DISTINCT did) AS n_distinct_positive_dids,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time
FROM positives;
""").fetchdf()


# CONTEGGI PER POSITIVO E LABEL
#
# Logica:
# - leggo solo gli eventi ufficiali, post-level, delle 5 label;
# - considero solo gli eventi dentro la utility window del positivo;
# - per ogni post × label tengo lo stato più recente nella finestra;
# - conto solo se lo stato più recente è added: neg = false.
#
# È sufficiente leggere dal 1 marzo al 22 marzo perché:
# - i takedown positivi sono nella finestra 11-21 marzo;
# - la utility window è di 10 giorni precedenti.


con.execute(f"""
CREATE TEMP TABLE counts_by_positive AS
WITH selected_events AS (
    SELECT
        split_part(uri, '/', 3) AS post_author_did,
        uri,
        val AS label_type,
        CAST(ts AS TIMESTAMP) AS label_ts,
        lower(neg) AS neg
    FROM read_csv(
        {sql_paths},
        header = true,
        delim = ',',
        all_varchar = true
    )
    WHERE ts >= '2026-03-01'
      AND ts <  '2026-03-22'
      AND src = '{OFFICIAL_LABELER}'
      AND uri LIKE 'at://%/app.bsky.feed.post/%'
      AND val IN ({labels_sql})
      AND lower(neg) IN ('true', 'false')
),
events_in_positive_window AS (
    SELECT
        p.did,
        p.event_time,
        e.uri,
        e.label_type,
        arg_max(e.neg, e.label_ts) AS latest_neg
    FROM selected_events e
    INNER JOIN positives p
        ON e.post_author_did = p.did
       AND e.label_ts >= p.event_time - INTERVAL '10 days'
       AND e.label_ts <  p.event_time
    GROUP BY
        p.did,
        p.event_time,
        e.uri,
        e.label_type
)
SELECT
    did,
    event_time,
    label_type,
    COUNT(*) AS n_labeled_posts
FROM events_in_positive_window
WHERE latest_neg = 'false'
GROUP BY did, event_time, label_type;
""")


# OUTPUT 1: QUANTI POSITIVI HANNO CIASCUNA LABEL


summary = con.execute("""
WITH wanted_labels(label_type, label_order) AS (
    VALUES
        ('porn', 1),
        ('sexual', 2),
        ('nudity', 3),
        ('sexual-figurative', 4),
        ('rude', 5)
),
n_positive AS (
    SELECT COUNT(*) AS n
    FROM positives
)
SELECT
    w.label_type,
    COUNT(c.did) AS n_positive_with_label_10d,
    ROUND(100.0 * COUNT(c.did) / n_positive.n, 3)
        AS pct_positive_with_label_10d,
    COALESCE(SUM(c.n_labeled_posts), 0)
        AS n_labeled_posts_10d,
    ROUND(AVG(c.n_labeled_posts), 3)
        AS avg_posts_among_labeled_positives,
    ROUND(MEDIAN(c.n_labeled_posts), 3)
        AS median_posts_among_labeled_positives,
    COALESCE(MAX(c.n_labeled_posts), 0)
        AS max_posts_one_positive
FROM wanted_labels w
CROSS JOIN n_positive
LEFT JOIN counts_by_positive c
    ON c.label_type = w.label_type
GROUP BY w.label_type, w.label_order, n_positive.n
ORDER BY w.label_order;
""").fetchdf()


# OUTPUT 2: DISTRIBUZIONE DEI SOLI POSITIVI LABELIZZATI


distribution = con.execute("""
WITH bucketed AS (
    SELECT
        label_type,
        CASE
            WHEN n_labeled_posts = 1 THEN '1'
            WHEN n_labeled_posts = 2 THEN '2'
            WHEN n_labeled_posts BETWEEN 3 AND 5 THEN '3-5'
            WHEN n_labeled_posts BETWEEN 6 AND 10 THEN '6-10'
            WHEN n_labeled_posts BETWEEN 11 AND 25 THEN '11-25'
            WHEN n_labeled_posts BETWEEN 26 AND 50 THEN '26-50'
            WHEN n_labeled_posts BETWEEN 51 AND 100 THEN '51-100'
            ELSE '101+'
        END AS posts_bucket,
        CASE
            WHEN n_labeled_posts = 1 THEN 1
            WHEN n_labeled_posts = 2 THEN 2
            WHEN n_labeled_posts BETWEEN 3 AND 5 THEN 3
            WHEN n_labeled_posts BETWEEN 6 AND 10 THEN 4
            WHEN n_labeled_posts BETWEEN 11 AND 25 THEN 5
            WHEN n_labeled_posts BETWEEN 26 AND 50 THEN 6
            WHEN n_labeled_posts BETWEEN 51 AND 100 THEN 7
            ELSE 8
        END AS bucket_order
    FROM counts_by_positive
)
SELECT
    label_type,
    posts_bucket,
    COUNT(*) AS n_positive_accounts,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY label_type),
        3
    ) AS pct_among_labeled_positives
FROM bucketed
GROUP BY label_type, posts_bucket, bucket_order
ORDER BY
    CASE label_type
        WHEN 'porn' THEN 1
        WHEN 'sexual' THEN 2
        WHEN 'nudity' THEN 3
        WHEN 'sexual-figurative' THEN 4
        WHEN 'rude' THEN 5
        ELSE 99
    END,
    bucket_order;
""").fetchdf()

# OUTPUT AGGIUNTIVO: POSITIVI CON ALMENO UNA DELLE LABEL SELEZIONATE

any_label_summary = con.execute("""
WITH positives_with_any_label AS (
    SELECT DISTINCT
        did,
        event_time
    FROM counts_by_positive
),
totals AS (
    SELECT COUNT(*) AS n_positive_total
    FROM positives
),
labeled AS (
    SELECT COUNT(*) AS n_positive_with_any_label_10d
    FROM positives_with_any_label
)
SELECT
    totals.n_positive_total,
    labeled.n_positive_with_any_label_10d,
    totals.n_positive_total - labeled.n_positive_with_any_label_10d
        AS n_positive_without_any_label_10d,
    ROUND(
        100.0 * labeled.n_positive_with_any_label_10d / totals.n_positive_total,
        3
    ) AS pct_positive_with_any_label_10d
FROM totals
CROSS JOIN labeled;
""").fetchdf()

# STAMPA


print("\n POSITIVI CON ALMENO UNA LABEL POST-LEVEL UFFICIALE ATTIVA NEI 10 GIORNI ")
print(any_label_summary.to_string(index=False))

print("\n LABEL POST-LEVEL UFFICIALI ATTIVE NEI 10 GIORNI PRIMA DEL TAKEDOWN ")
print(summary.to_string(index=False))

print("\n DISTRIBUZIONE DEL NUMERO DI POST LABELIZZATI TRA I POSITIVI CON LABEL ")
if distribution.empty:
    print("Nessun positivo presenta le label selezionate nella finestra considerata.")
else:
    print(distribution.to_string(index=False))

con.close()
PY